# ⚡ GLM Residual Boosting: Blending Convex Linear Margins with Trees (Kaggle PGS S6E11)

Welcome to the **GLM Residual Boosting Notebook** for **Kaggle Playground Series Season 6 Episode 11 (Predicting Credit Card Fraud)**.

### 🔬 The Mathematics & Rationale
1. Credit card fraud detection is characterized by **extreme class imbalance (~1.67% positives)** and high feature dimensionality.
2. An L2-regularized **Generalized Linear Model (Logistic Regression)** finds a globally convex, well-calibrated hyperplane:
   $$z_i = \mathbf{w}^T \mathbf{x}_i + b = \log \left( \frac{p_i}{1 - p_i} \right)$$
   However, linear models cannot capture non-linear feature interactions (e.g. `amount_usd` $\times$ `merchant_risk_score`).
3. Standard GBDTs (Decision Trees) easily capture non-linear thresholds, but can overfit the majority negative class in extreme imbalance.
4. **GLM Residual Boosting Solution**:
   We feed the linear log-odds $z_i$ directly into LightGBM as `init_score`:
   ```python
   trn_data = lgb.Dataset(X_tr, label=y_tr, init_score=z_tr)
   val_data = lgb.Dataset(X_va, label=y_va, init_score=z_va, reference=trn_data)
   ```
   LightGBM's trees are regularized to predict $f(\mathbf{x}) = 0$ unless there is a genuine non-linear residual that improves binary cross-entropy:
   $$P(y_i = 1 | \mathbf{x}_i) = \sigma\left( z_i + \sum_{m=1}^M \eta f_m(\mathbf{x}_i) \right)$$
5. This yields faster convergence, reduced tree variance, and higher generalization performance!

In [ ]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from scipy.special import expit  # Sigmoid function

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f"LightGBM Version: {lgb.__version__}")

In [ ]:
class Config:
    RANDOM_STATE = 42
    N_SPLITS = 5
    
    # Path resolution
    if os.path.exists('/kaggle/input/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    elif os.path.exists('/kaggle/input/competitions/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    else:
        DATA_DIR = Path('../../dataset')
        if not (DATA_DIR / 'train.csv').exists():
            DATA_DIR = Path('pgs-s6e11/dataset')
        OUTPUT_DIR = Path('../../submission')
        if not OUTPUT_DIR.exists():
            OUTPUT_DIR = Path('pgs-s6e11/submission')

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data Directory:   {Config.DATA_DIR}")
print(f"Output Directory: {Config.OUTPUT_DIR}")

In [ ]:
train_df = pd.read_csv(Config.DATA_DIR / 'train.csv')
test_df = pd.read_csv(Config.DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(Config.DATA_DIR / 'sample_submission.csv')

features = [c for c in train_df.columns if c not in ['id', 'is_fraud']]
cat_cols = ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']
num_cols = [c for c in features if c not in cat_cols]
bool_cols = [
    'is_foreign_transaction', 'is_new_merchant', 'used_vpn',
    'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt'
]

# Convert categorical variables to category dtype for LightGBM
X_gbdt = train_df[features].copy()
X_test_gbdt = test_df[features].copy()
for col in cat_cols:
    X_gbdt[col] = X_gbdt[col].astype('category')
    X_test_gbdt[col] = X_test_gbdt[col].astype('category')

for b in bool_cols:
    X_gbdt[b] = X_gbdt[b].astype(int)
    X_test_gbdt[b] = X_test_gbdt[b].astype(int)

y = train_df['is_fraud'].astype(int).values

print(f"Dataset Loaded: {len(train_df):,} Train rows, {len(test_df):,} Test rows")
print(f"Target distribution: {np.bincount(y)} (Fraud Rate: {y.mean()*100:.3f}%)")

In [ ]:
skf = StratifiedKFold(n_splits=Config.N_SPLITS, shuffle=True, random_state=Config.RANDOM_STATE)

oof_lr_probs = np.zeros(len(train_df), dtype=np.float64)
oof_boost_probs = np.zeros(len(train_df), dtype=np.float64)

test_lr_margin = np.zeros(len(test_df), dtype=np.float64)
test_boost_margin = np.zeros(len(test_df), dtype=np.float64)

lr_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

print("=" * 70)
print("Training 5-Fold GLM Residual Boosting Pipeline...")
print("=" * 70)

start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, y), 1):
    fold_t0 = time.time()
    
    # --- Step 1: Fit Logistic Regression Base Classifier ---
    X_tr_lr = lr_preprocessor.fit_transform(train_df.iloc[train_idx][features])
    X_va_lr = lr_preprocessor.transform(train_df.iloc[val_idx][features])
    X_te_lr = lr_preprocessor.transform(test_df[features])
    y_tr, y_va = y[train_idx], y[val_idx]
    
    lr = LogisticRegression(C=0.1, max_iter=1000, random_state=Config.RANDOM_STATE)
    lr.fit(X_tr_lr, y_tr)
    
    # Compute linear margins (decision function = log-odds z)
    z_tr = lr.decision_function(X_tr_lr)
    z_va = lr.decision_function(X_va_lr)
    z_te = lr.decision_function(X_te_lr)
    
    # Store LR baseline probabilities
    oof_lr_probs[val_idx] = expit(z_va)
    auc_lr_fold = roc_auc_score(y_va, oof_lr_probs[val_idx])
    
    # --- Step 2: Fit LightGBM Trees on Residuals using init_score ---
    X_tr_g, X_va_g = X_gbdt.iloc[train_idx], X_gbdt.iloc[val_idx]
    
    trn_data = lgb.Dataset(X_tr_g, label=y_tr, init_score=z_tr)
    val_data = lgb.Dataset(X_va_g, label=y_va, init_score=z_va, reference=trn_data)
    
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': 0.03,
        'num_leaves': 4,
        'max_depth': 3,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': Config.RANDOM_STATE + fold,
        'n_estimators': 1500,
        'verbose': -1,
        'n_jobs': -1
    }
    
    evals_result = {}
    booster = lgb.train(
        params,
        trn_data,
        valid_sets=[trn_data, val_data],
        valid_names=['train', 'val'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.record_evaluation(evals_result)
        ]
    )
    
    # Predict residual margins from trees (raw_score=True)
    tree_margin_va = booster.predict(X_va_g, raw_score=True)
    tree_margin_te = booster.predict(X_test_gbdt, raw_score=True)
    
    # Combine: total log-odds = z + tree_residual
    total_margin_va = z_va + tree_margin_va
    oof_boost_probs[val_idx] = expit(total_margin_va)
    
    total_margin_te = z_te + tree_margin_te
    test_boost_margin += total_margin_te / Config.N_SPLITS
    
    auc_boost_fold = roc_auc_score(y_va, oof_boost_probs[val_idx])
    pr_boost_fold = average_precision_score(y_va, oof_boost_probs[val_idx])
    
    gain = auc_boost_fold - auc_lr_fold
    print(f"Fold {fold} | LR AUC: {auc_lr_fold:.5f} -> Boosted AUC: {auc_boost_fold:.5f} (+{gain:.5f}) | PR-AUC: {pr_boost_fold:.5f} ({time.time()-fold_t0:.1f}s)")

total_cv_time = time.time() - start_time
print("=" * 70)
print(f"GLM Residual Boosting Completed in {total_cv_time:.1f}s!")

In [ ]:
overall_lr_auc = roc_auc_score(y, oof_lr_probs)
overall_boost_auc = roc_auc_score(y, oof_boost_probs)
overall_boost_pr = average_precision_score(y, oof_boost_probs)

print("-" * 60)
print(f"Baseline Logistic Regression OOF ROC-AUC: {overall_lr_auc:.5f}")
print(f"GLM Residual Boosting OOF ROC-AUC:       {overall_boost_auc:.5f} (+{overall_boost_auc - overall_lr_auc:.5f})")
print(f"GLM Residual Boosting OOF PR-AUC:        {overall_boost_pr:.5f}")
print("-" * 60)

np.save(Config.OUTPUT_DIR / 'oof_glm_boost.npy', oof_boost_probs)
print("Saved oof_glm_boost.npy for ensembling.")

# Plot ROC Curves
plt.figure(figsize=(10, 5))
fpr_lr, tpr_lr, _ = roc_curve(y, oof_lr_probs)
fpr_boost, tpr_boost, _ = roc_curve(y, oof_boost_probs)

plt.plot(fpr_lr, tpr_lr, color='gray', linestyle='--', label=f'Baseline LR (AUC = {overall_lr_auc:.5f})')
plt.plot(fpr_boost, tpr_boost, color='#1565c0', lw=2, label=f'GLM Residual Boosted (AUC = {overall_boost_auc:.5f})')
plt.plot([0, 1], [0, 1], color='lightgray', linestyle=':')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('GLM Residual Boosting ROC Improvement')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('glm_boost_roc.png', dpi=150)
plt.show()

In [ ]:
test_final_probs = expit(test_boost_margin)

submission = pd.DataFrame({
    'id': test_df['id'],
    'is_fraud': test_final_probs
})

assert submission.shape == sample_sub.shape, f"Shape mismatch: {submission.shape} vs {sample_sub.shape}"
assert list(submission.columns) == ['id', 'is_fraud'], f"Column mismatch: {submission.columns}"
assert not submission.isna().any().any(), "Submission contains null/NaN values!"
assert (submission['is_fraud'] >= 0.0).all() and (submission['is_fraud'] <= 1.0).all(), "Predictions out of [0, 1] range!"
assert (submission['id'] == sample_sub['id']).all(), "ID alignment mismatch!"

sub_path = Config.OUTPUT_DIR / 'glm_boost_submission.csv'
submission.to_csv(sub_path, index=False)
submission.to_csv(Config.OUTPUT_DIR / 'submission.csv', index=False)

print(f"Submission successfully generated and verified at: {sub_path}")
print(f"File size: {os.path.getsize(sub_path) / (1024*1024):.2f} MB")
print("\nFirst 10 submission rows:")
print(submission.head(10))